# week 1 [ GENERIC ALGORITHM]
Employee shift scheduling

In [1]:
import random

employees = 6
shifts = 3
POP, GEN, MUT = 20, 50, 0.1

def fitness(schedule):
    counts = [schedule.count(i) for i in range(shifts)]
    return sum(abs(x - 2) for x in counts)  

population = [
    [random.randrange(shifts) for _ in range(employees)]
    for _ in range(POP)
]

for _ in range(GEN):
    population.sort(key=fitness)
    parents = population[:10]
    new = parents.copy()

    while len(new) < POP:
        p1, p2 = random.sample(parents, 2)
        point = random.randint(1, employees - 1)
        child = p1[:point] + p2[point:]

        if random.random() < MUT:
            child[random.randrange(employees)] = random.randrange(shifts)

        new.append(child)

    population = new

best = min(population, key=fitness)

print("Best Schedule:", best)
print("Conflicts:", fitness(best))

Best Schedule: [0, 1, 0, 1, 2, 2]
Conflicts: 0


# week 2 [PARICLE SWARM OPTIMIZATION ]
[FINDING NEAREST SCHOOL]|

In [4]:
import random
import math
student = (50, 50)
def fitness(x, y):
    return math.sqrt((x - student[0])**2 + (y - student[1])**2)

n = 10
iterations = 30
w = 0.5
c1 = 1.5
c2 = 1.5
particles = []

for i in range(n):
    x = random.uniform(0, 100)
    y = random.uniform(0, 100)
    particles.append({
        "x": x,
        "y": y,
        "vx": random.uniform(-1, 1),
        "vy": random.uniform(-1, 1),
        "pbest": (x, y),
        "best": fitness(x, y) })

gbest = min(particles, key=lambda p: p["best"])
gx, gy = gbest["pbest"]
for _ in range(iterations):
    for p in particles:
        f = fitness(p["x"], p["y"])

        if f < p["best"]:
            p["pbest"] = (p["x"], p["y"])
            p["best"] = f
    gbest = min(particles, key=lambda p: p["best"])
    gx, gy = gbest["pbest"]
    for p in particles:
        r1 = random.random()
        r2 = random.random()
        p["vx"] = (
            w * p["vx"]
            + c1 * r1 * (p["pbest"][0] - p["x"])
            + c2 * r2 * (gx - p["x"]))
        p["vy"] = (
            w * p["vy"]
            + c1 * r1 * (p["pbest"][1] - p["y"])
            + c2 * r2 * (gy - p["y"]) )
        p["x"] += p["vx"]
        p["y"] += p["vy"]

print("Nearest School Location:", (round(gx, 2), round(gy, 2)))
print("Distance:", round(fitness(gx, gy), 2))

Nearest School Location: (50.0, 50.0)
Distance: 0.0


# week 3 [ OPTIMIZATION VIA GENE EXPRESSION ALGORITHM]|
[DELIVERY ROUTE OPTIMIZATION]

In [2]:
import random

dist = [
    [0,10,15,20,25],
    [10,0,35,25,30],
    [15,35,0,30,20],
    [20,25,30,0,15],
    [25,30,20,15,0]
]

N, POP, GEN, MUT = 5, 20, 100, 0.1

def distance(r):
    return sum(dist[r[i]][r[(i+1) % N]] for i in range(N))

population = []
for _ in range(POP):
    r = list(range(N))
    random.shuffle(r)
    population.append(r)

for _ in range(GEN):
    population.sort(key=distance)
    parents = population[:10]
    new = parents.copy()

    while len(new) < POP:
        p1, p2 = random.sample(parents, 2)
        point = random.randint(1, N-2)
        child = p1[:point] + [x for x in p2 if x not in p1[:point]]

        if random.random() < MUT:
            a, b = random.sample(range(N), 2)
            child[a], child[b] = child[b], child[a]

        new.append(child)

    population = new

best = min(population, key=distance)
print("Best Route:", best)
print("Minimum Distance:", distance(best))

Best Route: [1, 0, 2, 4, 3]
Minimum Distance: 85


# week 4 Ant colony optimization 

In [2]:
import random
def ant_colony_tsp(distance, num_ants=10, iterations=100,
                   alpha=1, beta=2, rho=0.5, Q=100):
    n = len(distance)
    pheromone = [[1.0 for _ in range(n)] for _ in range(n)]
    best_tour = None
    best_distance = float("inf")
    for iteration in range(iterations):
        all_tours = []
        for ant in range(num_ants):
            start = random.randrange(n)
            tour = [start]
            unvisited = list(range(n))
            unvisited.remove(start)
            
            while unvisited:
                current = tour[-1]
                weights = []
                for city in unvisited:
                    tau = pheromone[current][city] ** alpha
                    eta = (1 / distance[current][city]) ** beta
                    weights.append(tau * eta)

                total = sum(weights)
                probabilities = [w / total for w in weights]
                next_city = random.choices(
                    unvisited, weights=probabilities, k=1
                )[0]
                tour.append(next_city)
                unvisited.remove(next_city)
            tour.append(start)
            length = sum(
                distance[tour[i]][tour[i + 1]]
                for i in range(n) )
            all_tours.append((tour, length))

            if length < best_distance:
                best_distance = length
                best_tour = tour[:]
        for i in range(n):
            for j in range(n):
                pheromone[i][j] *= (1 - rho)
        for tour, length in all_tours:
            deposit = Q / length
            for i in range(n):
                a = tour[i]
                b = tour[i + 1]
                pheromone[a][b] += deposit
                pheromone[b][a] += deposit
    return best_tour, best_distance
distance = [
    [0,  10, 15, 20, 18],
    [10,  0, 35, 25, 12],
    [15, 35,  0, 30, 10],
    [20, 25, 30,  0, 14],
    [18, 12, 10, 14,  0]]
tour, length = ant_colony_tsp(distance)
print("Best Tour:", " -> ".join(map(str, tour)))
print("Minimum Distance:", length)

Best Tour: 3 -> 1 -> 0 -> 2 -> 4 -> 3
Minimum Distance: 74
